# Word Mover’s Distance(WMD)

NLP연구쪽에서 문서 유사도 구하는 방법으로 잘 쓰인다고 함. (논문 연구 결과 다른 방법론보다 성능이 좋다고 함)\
기본적으로 word2vec을 베이스로 하고, word2vec을 사용해서 word간의 euclide distance를 구함!

- 피험자 1명, seed 단어 1개

피험자가 대답한 30개의 연속적인 단어 그룹 - target word(money, friend, family)의 유사도를 기반으로 wmdistance를 잰다.

--> 그럼 3개의 distance값이 나옴. 그 중 가장 값이 작은 것이 거리가 가깝다. 즉, target word와 유사하다.

wmd 설명 참조
- https://sy-programmingstudy.tistory.com/14
- https://www.youtube.com/watch?v=zFnrq5SmBdg

In [37]:
import os
import platform

import pandas as pd
import numpy as np
from gensim.models import KeyedVectors

In [27]:
os_system = platform.system() # 맥북은 Darwin, 윈도우는 Windows

# 현재 프로젝트 폴더 위치 지정. os.getcwd()는 지금 코드 실행하는 현 위치를 출력해줍니다.
research1_dir = os.getcwd()
model_path = '\\..\\pretrained\\GoogleNews-vectors-negative300.bin' if os_system == 'Windows' else '/../pretrained/GoogleNews-vectors-negative300.bin'

# data/processed 폴더 위치 지정
processed_data_dir = research1_dir + ('\\data\\processed\\' if os_system == 'Windows' else '/data/processed/')

In [28]:
# word2vec model 로딩
word2vec_model = KeyedVectors.load_word2vec_format(research1_dir + model_path, binary=True)

In [65]:
# 테이블 읽어오기
tbl_data = pd.read_csv(processed_data_dir + 'merged_data.csv')
tbl_data[0:3]

,subject,tear1,tear2,tear3,tear4,tear5,tear6,tear7,tear8,tear9,...,abuse31,abuse32,abuse33,abuse34,abuse35,abuse36,abuse37,abuse38,abuse39,abuse40
0,1,sadness,depressed,coolness,annoyance,failure,upset,conflict,farewell,heartache,...,a strange woman,do not ask assault,crime,detective,drama,kwon ryongi narsha,history drama,king seondeok,female,subjectivity
1,2,sadness,sob,sad ending,movie,helminth,bug,apple,fruit,melon,...,announcement,announcement,recruitment,participant,experiment,lab,research complex,gnme,major,student council
2,3,masterpiece,world,travel,carrier,airport,airplane,sky,cloud,happiness,...,smell,sensitive,myself,singularity,geezer,scientist,difficulty,headache,tylenol,labor pains


In [30]:
seed_words = ['abuse', 'tear', 'mirror', 'family'] # 학대, 눈물, 거울, 가족
target_words = ['money', 'friend', 'relationships', 'family'] 
n_respond_words = 40 # 하나의 시드당 40개의 단어 응답
n_subject = len(tbl_data) # 350
n_dim_of_vector = 300

In [82]:
# distance_seedword_targetword 컬럼 미리 생성(빈 값)
for seed_word in seed_words:
    for target_word in target_words:
        column_name = f'distance_{seed_word}_{target_word}'
        tbl_data[column_name] = np.nan

tbl_data.columns


Index(['subject', 'tear1', 'tear2', 'tear3', 'tear4', 'tear5', 'tear6',
       'tear7', 'tear8', 'tear9',
       ...
       'distance_tear_relationships', 'distance_tear_family',
       'distance_mirror_money', 'distance_mirror_friend',
       'distance_mirror_relationships', 'distance_mirror_family',
       'distance_family_money', 'distance_family_friend',
       'distance_family_relationships', 'distance_family_family'],
      dtype='object', length=177)

In [78]:
# 응답 단어 컬럼 목록 (예: 'abuse1', 'abuse2', ... , 'abuse40', ... , 'family40')
word_columns = [f'{seed_word}{i}' for seed_word in seed_words for i in range(1, n_respond_words + 1)]
# word_columns

In [83]:
for i_subject in range(n_subject):
    # print('subject: ', i_subject)

    for seed_word in seed_words: # abuse, tear, mirror, family
        # 각 피험자의 응답 문서를 40개씩 단어 리스트로 변환
        seed_response_words = [tbl_data.iloc[i_subject][column] for column in word_columns if column.startswith(seed_word)]
        # nan값 걸러내기
        seed_response_words = [word for word in seed_response_words if not isinstance(word, float) or not np.isnan(word)] 
        if len(seed_response_words) > 0: # 애초에 값이 0인 피험자 데이터들은 건너뛰도록 함
            for target_word in target_words: # money, friend, relationships, family
                try:
                    # 응답 40개 단어 - target 단어의 WMD 계산
                    wmdistance_value = word2vec_model.wmdistance([target_word], seed_response_words)

                    # 계산한 WMD를 해당 테이블 위치에 저장
                    tbl_data.at[i_subject, f'distance_{seed_word}_{target_word}'] = wmdistance_value
                    print(f'distance_{seed_word}_{target_word}: {wmdistance_value}')
                except Exception as e:
                    print(f"An error occurred for subject {i_subject}: {str(e)}")
                    continue


distance_abuse_money: 1.31953949423936
distance_abuse_friend: 1.2982317705356576
distance_abuse_relationships: 1.3431514634824482
distance_abuse_family: 1.3055505415503348
distance_tear_money: 1.3003448904017942
distance_tear_friend: 1.2722680878429695
distance_tear_relationships: 1.3327086021723547
distance_tear_family: 1.2378991882648407
distance_mirror_money: 1.3321392210488912
distance_mirror_friend: 1.3280419484603188
distance_mirror_relationships: 1.3909049298940857
distance_mirror_family: 1.3179760703200247
distance_family_money: 1.3303716118931144
distance_family_friend: 1.3047254807834636
distance_family_relationships: 1.3419701079597117
distance_family_family: 1.3262192316768093
distance_abuse_money: 1.2896375992034186
distance_abuse_friend: 1.3589958924933936
distance_abuse_relationships: 1.372745272692484
distance_abuse_family: 1.3553638653368347
distance_tear_money: 1.3773429089133804
distance_tear_friend: 1.3376540423639647
distance_tear_relationships: 1.366239257272977
d

In [85]:
# 단어 있는 버전 csv 저장
tbl_data.to_csv(processed_data_dir + 'distance.csv', index=None)

# 단어 컬럼들 드롭
drop_columns = tbl_data.columns[1:161]
tbl_data = tbl_data.drop(drop_columns, axis='columns')

# 단어 없이 coherence만 있는 버전 csv 저장
tbl_data.to_csv(processed_data_dir + 'distance_without_words.csv', index=None)